In [1]:
import time
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.home() / "icidea_llm_ids"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

LABEL_MAP = {0: "NORMAL", 1: "DOS", 2: "FUZZY", 3: "GEAR", 4: "RPM"}

# Load Section 3 artifact
t0 = time.time()
df = pd.read_parquet(ARTIFACTS_DIR / "section3_loaded_data.parquet")
print(f"✓ Loaded {len(df):,} rows in {time.time()-t0:.2f}s")
print(f"  Columns: {list(df.columns)}")
print(f"  Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

✓ Loaded 1,000,000 rows in 5.31s
  Columns: ['timestamp', 'ID', 'DLC', 'DATA0', 'DATA1', 'DATA2', 'DATA3', 'DATA4', 'DATA5', 'DATA6', 'DATA7', 'class']
  Memory: 96.0 MB


In [2]:
rename_map = {
    "ID":    "can_id",
    "DLC":   "dlc",
    **{f"DATA{i}": f"data{i}" for i in range(8)},
    "class": "label",
}

df = df.rename(columns=rename_map)

print("Renamed columns:")
print(list(df.columns))

Renamed columns:
['timestamp', 'can_id', 'dlc', 'data0', 'data1', 'data2', 'data3', 'data4', 'data5', 'data6', 'data7', 'label']


In [3]:
df["timestamp"] = df["timestamp"].astype("float64")
df["can_id"]    = df["can_id"].astype("uint16")
df["dlc"]       = df["dlc"].astype("uint8")
for i in range(8):
    df[f"data{i}"] = df[f"data{i}"].astype("uint8")
df["label"] = df["label"].astype("uint8")

print("Column dtypes after standardization:")
print(df.dtypes.to_string())
print(f"\nMemory after type enforcement: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Column dtypes after standardization:
timestamp    float64
can_id        uint16
dlc            uint8
data0          uint8
data1          uint8
data2          uint8
data3          uint8
data4          uint8
data5          uint8
data6          uint8
data7          uint8
label          uint8

Memory after type enforcement: 20.0 MB


In [4]:
CANONICAL_COLS = [
    "timestamp", "can_id", "dlc",
    "data0", "data1", "data2", "data3",
    "data4", "data5", "data6", "data7",
    "label"
]

df = df[CANONICAL_COLS]

# Sort within each class by timestamp to preserve temporal ordering
# (critical for windowing in Section 6)
df = df.sort_values(["label", "timestamp"]).reset_index(drop=True)

print("Column order confirmed:")
print(list(df.columns))
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nTimestamp range per class:")
for lbl, name in LABEL_MAP.items():
    subset = df[df["label"] == lbl]
    print(f"  {name:7s}  {subset['timestamp'].min():.3f} → {subset['timestamp'].max():.3f}")

Column order confirmed:
['timestamp', 'can_id', 'dlc', 'data0', 'data1', 'data2', 'data3', 'data4', 'data5', 'data6', 'data7', 'label']

First 5 rows:
      timestamp  can_id  dlc  data0  data1  data2  data3  data4  data5  data6  \
0  1.479121e+09     848    8      5     40    132    102    109      0      0   
1  1.479121e+09     399    8    254     54      0      0      0     60      0   
2  1.479121e+09     672    8     96      0    107     29      1      4    221   
3  1.479121e+09     809    8    135    185    126     20     18     32      0   
4  1.479121e+09    1087    8      0     64     96    255     90    108      8   

   data7  label  
0    162      0  
1      0      0  
2      0      0  
3     20      0  
4      0      0  

Timestamp range per class:
  NORMAL   1479121434.850 → 1479121941.286
  DOS      1478198377.185 → 1478200651.438
  FUZZY    1478195722.775 → 1478198177.328
  GEAR     1478193191.268 → 1478195126.677
  RPM      1478191030.967 → 1478192970.836


In [5]:
print("Running schema validation...")

assert list(df.columns) == CANONICAL_COLS, "Column order mismatch"
assert df["can_id"].max() <= 2047, f"CAN ID out of 11-bit range: {df['can_id'].max()}"
assert df["dlc"].max() <= 8, f"DLC out of range: {df['dlc'].max()}"
for i in range(8):
    assert df[f"data{i}"].min() >= 0, f"data{i} has negative values"
    assert df[f"data{i}"].max() <= 255, f"data{i} out of uint8 range"
assert set(df["label"].unique()) == {0, 1, 2, 3, 4}, \
    f"Unexpected labels: {df['label'].unique()}"
assert df["label"].value_counts().min() >= 190_000, \
    f"Class imbalance detected: {df['label'].value_counts().to_dict()}"

print("✓ All schema assertions passed")
print(f"  Rows:    {len(df):,}")
print(f"  Classes: {sorted(df['label'].unique().tolist())}")
print(f"  CAN ID range: {df['can_id'].min()} – {df['can_id'].max()}")
print(f"  DLC range:    {df['dlc'].min()} – {df['dlc'].max()}")

Running schema validation...
✓ All schema assertions passed
  Rows:    1,000,000
  Classes: [0, 1, 2, 3, 4]
  CAN ID range: 0 – 2047
  DLC range:    2 – 8


In [6]:
artifact_path = ARTIFACTS_DIR / "section4_standardized_data.parquet"
df.to_parquet(artifact_path, index=False)
size_mb = artifact_path.stat().st_size / 1e6
print(f"✓ Saved to {artifact_path}")
print(f"  Size: {size_mb:.1f} MB")
print(f"  Schema: {list(df.columns)}")
print(f"  Sections 5+ will load from this file.")

✓ Saved to /Users/deepakpatnaik/icidea_llm_ids/artifacts/section4_standardized_data.parquet
  Size: 9.0 MB
  Schema: ['timestamp', 'can_id', 'dlc', 'data0', 'data1', 'data2', 'data3', 'data4', 'data5', 'data6', 'data7', 'label']
  Sections 5+ will load from this file.
